# Gift Recommendation LLM Evaluation

This notebook evaluates different LLM models for gift recommendation generation.

## Metrics Tracked
- **latency_ms**: Time per prompt
- **tokens_in / tokens_out**: Inference cost basis
- **quality_score**: LLM-as-a-judge or rubric scoring
- **monthly_estimate**: Cost for 2B requests

## Models Evaluated
- OpenAI compatible API (e.g., OpenAI, Token Factory)
- Self-hosted vLLM (7B, 13B)

In [1]:
%load_ext autoreload
%autoreload 2


import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "prototype" / "src"))

from src.evaluation import (
    evaluate_quality_with_judge,
    setup_mlflow,
)
from src.generators import generate_gift_recommendation
from src.model_config import get_judge_client, get_models_for_evaluation
from src.prompts import (
    generate_gift_quality_judge_prompt,
)
from src.test_samples import get_test_profiles
from src.utils import load_env_from_repo_root

# Load .env file from repository root
load_env_from_repo_root('.env', override=True)

In [2]:
setup_mlflow("gift_recommendation_eval")

MLflow tracking URI: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud
MLflow experiment: <Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1765790976980, experiment_id='1', last_update_time=1765790976980, lifecycle_stage='active', name='gift_recommendation_eval', tags={}>


In [3]:

# Get all available models
models_to_evaluate = get_models_for_evaluation(model_names=[
    "gpt-4o-mini",
    "openai-gpt-3.5-turbo",
    "tf/gpt-oss-20b",
    "tf/DeepSeek-R1-0528",
    ])

# Or get specific models only (uncomment to use):
# models_to_evaluate = get_models_for_evaluation(model_names=["gpt-4o-mini", "tf/gpt-oss-20b"])

In [4]:
print("Smoke-testing model availability...")

for m in models_to_evaluate:
    model_name = m["name"]
    try:
        text, metrics = m["client"].generate(
            prompt="ping",
            temperature=0.0,
            max_tokens=8,
        )
        print(f"✅ {model_name}: reachable (latency_ms={metrics.get('latency_ms'):.1f}, tokens_out={metrics.get('tokens_out')})")
    except Exception as e:
        print(f"❌ {model_name}: {e}")

Smoke-testing model availability...
✅ gpt-4o-mini: reachable (latency_ms=916.7, tokens_out=8)
✅ openai-gpt-3.5-turbo: reachable (latency_ms=792.5, tokens_out=3)
✅ tf/gpt-oss-20b: reachable (latency_ms=295.1, tokens_out=8)
✅ tf/DeepSeek-R1-0528: reachable (latency_ms=721.4, tokens_out=8)


### Judge Client - gpt-5.2

In [5]:

judge_client = get_judge_client()


In [6]:
# # Test judge

# _kid_profile = get_test_profiles()[0]
# _client = models_to_evaluate[0]['client']

# test_prompt = generate_gift_recommendation(
#     _client, _kid_profile, temperature=0.7, max_tokens=300
#                 )
# test_response, _ = _client.generate(test_prompt, temperature=0.8, max_tokens=300)

# # judge_response, _ = judge_client.generate(test_judge_prompt, temperature=0.0)
# test_judge_prompt = generate_gift_quality_judge_prompt(_kid_profile, test_response)
# evaluate_quality_with_judge(judge_client, test_judge_prompt)


## Evaluate Gift Recommendations

In [ ]:
import re
from pathlib import Path

import mlflow
import pandas as pd

test_profiles  = get_test_profiles()

results = []
agg_results = []

MLFLOW_AVAILABLE = bool(mlflow.get_tracking_uri())
# Enable automatic tracing for all OpenAI API calls.
# mlflow.openai.autolog()
mlflow.autolog()

# Evaluate all configured models
for model_config in models_to_evaluate:
    model_name = model_config["name"]
    client = model_config["client"]
    print(f"Evaluating model: {model_name}")

    with mlflow.start_run(run_name=f"{model_name}"):
        parent_run_id = mlflow.active_run().info.run_id if MLFLOW_AVAILABLE else None
        per_calls = []

        for kid_profile in test_profiles:
            try:
                gift_recommendation, metrics = generate_gift_recommendation(
                    client, kid_profile, temperature=0.7, max_tokens=300
                )

                # TODO: revise/remove
                # Format gift_recommendation as string for judge evaluation
                response_text = "GIFTS:\n" + "\n".join(f"- {gift}" for gift in gift_recommendation.gifts)
                response_text += f"\n\nRATIONALE:\n{gift_recommendation.rationale}"

                quality_score = None
                quality_rationale = None
                if judge_client:
                    judge_prompt = generate_gift_quality_judge_prompt(kid_profile, response_text)
                    quality_score, quality_rationale = evaluate_quality_with_judge(judge_client, judge_prompt)

                tokens_in = metrics.get("tokens_in", 0) or 0
                tokens_out = metrics.get("tokens_out", 0) or 0
                cost_in = round(tokens_in * model_config["cost_per_1m_tokens_in"], 1)
                cost_out = round(tokens_out * model_config["cost_per_1m_tokens_out"], 1)
                cost_total = round(cost_in + cost_out, 1)

                record = {
                    "model": model_name,
                    "kid_id": kid_profile.id,
                    "latency_ms": round(metrics.get("latency_ms", 0), 0),
                    "tokens_in": tokens_in,
                    "tokens_out": tokens_out,
                    "quality_score": quality_score,
                    "cost_in_1m": cost_in,
                    "cost_out_1m": cost_out,
                    "cost_total_1m": cost_total,
                }
                per_calls.append(record)
                results.append(record)

                if MLFLOW_AVAILABLE:
                    with mlflow.start_run(run_name=f"{kid_profile.id}", nested=True):
                        mlflow.log_metric("latency_ms", record["latency_ms"])
                        mlflow.log_metric("tokens_in", tokens_in)
                        mlflow.log_metric("tokens_out", tokens_out)
                        if quality_score is not None:
                            mlflow.log_metric("quality_score", quality_score)
                        mlflow.log_metric("cost_in_1m", cost_in)
                        mlflow.log_metric("cost_out_1m", cost_out)
                        mlflow.log_metric("cost_total_1m", cost_total)
                        mlflow.set_tags({
                            "kid_id": kid_profile.id,
                            "task": "gift_recommendation",
                        })
                        mlflow.log_dict(
                            {
                                "gifts": gift_recommendation.gifts,
                                "rationale": gift_recommendation.rationale,
                                "quality_score": quality_score,
                                "quality_rationale": quality_rationale,
                            }
                            , "gift_recommendation.json")
            except Exception as e:
                err_rec = {
                    "model": model_name,
                    "kid_id": kid_profile.id,
                    "error": str(e),
                }
                per_calls.append(err_rec)
                results.append(err_rec)

        df_model = pd.DataFrame([r for r in per_calls if "error" not in r])
        if not df_model.empty:
            agg = {
                "model": model_name,
                "latency_ms": round(df_model["latency_ms"].mean(), 0),
                "tokens_in": df_model["tokens_in"].mean(),
                "tokens_out": df_model["tokens_out"].mean(),
                "quality_score": df_model["quality_score"].mean(),
                "cost_in_1m": round(df_model["cost_in_1m"].mean(), 1),
                "cost_out_1m": round(df_model["cost_out_1m"].mean(), 1),
                "cost_total_1m": round(df_model["cost_total_1m"].mean(), 1),
                "calls": len(df_model),
            }
            agg_results.append(agg)

            out_dir = Path("data/evaluation")
            out_dir.mkdir(parents=True, exist_ok=True)
            model_name_path = re.sub(r'[^\w\-]', '-', model_name)
            out_path = out_dir / f"01_gift_rec_eval-{model_name_path}.csv"
            df_model.to_csv(out_path, index=False)
            print(f"Saved per-call results for {model_name} -> {out_path}")

            if MLFLOW_AVAILABLE:
                mlflow.log_metric("latency_ms", agg["latency_ms"])
                mlflow.log_metric("tokens_in", agg["tokens_in"])
                mlflow.log_metric("tokens_out", agg["tokens_out"])
                mlflow.log_metric("quality_score", agg["quality_score"])
                mlflow.log_metric("cost_in_1m", agg["cost_in_1m"])
                mlflow.log_metric("cost_out_1m", agg["cost_out_1m"])
                mlflow.log_metric("cost_total_1m", agg["cost_total_1m"])
                mlflow.log_metric("calls", agg["calls"])
                mlflow.set_tags({"task": "gift_recommendation"})
        else:
            print(f"No successful calls for model {model_name}")


Evaluating model: gpt-4o-mini
🏃 View run c5a39f2b-2e28-4b55-b1e9-b3bb9b60b742 at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1/runs/884a7cb83cae4f76b6e5d942641ba67a
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1
🏃 View run 56c7f4cd-f488-496e-a4ce-28917d5a970f at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1/runs/7a426fb3229540ffbcc4bf65d3e8a63d
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1
🏃 View run e889a5c1-fb59-41bf-9196-c38893789639 at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw.msp.eu-north1.nebius.cloud/#/experiments/1/runs/b7ff464e77df4767ae19045390762ba0
🧪 View experiment at: https://public-tracking-e00-q0ycj5wbge9njs0-p4e20s5hwjds743-mlflow.gw

In [8]:
# Final aggregated summary
df_results = pd.DataFrame(results)
df_agg = pd.DataFrame(agg_results)
print("Per-model aggregate summary:")
df_agg.round(3)

Per-model aggregate summary:


,model,latency_ms,tokens_in,tokens_out,quality_score,cost_in_1m,cost_out_1m,cost_total_1m,calls
0,gpt-4o-mini,2959.0,209.00,115.50,0.900,31.3,69.3,100.6,4
1,openai-gpt-3.5-turbo,1171.0,212.00,85.25,0.862,318.0,170.5,488.5,4
2,tf/gpt-oss-20b,1156.0,271.00,295.75,0.212,40.6,177.4,218.1,4
3,tf/DeepSeek-R1-0528,5726.0,213.25,106.00,0.888,170.6,254.4,425.0,4
